**BACKGROUND GENERATION**

**2 cells- Run 1st cell and restart session, After restarting, run Cell 2**

In [ ]:
# CELL 1: INSTALLATION ONLY
# 1. Clean out all conflicting Colab defaults
!pip uninstall -y cupy-cuda12x jax jaxlib opencv-python opencv-python-headless opencv-contrib-python pillow numpy --quiet

# 2. Install all strictly compatible versions in a single command
!pip install diffusers==0.29.2 transformers==4.45.0 accelerate safetensors torch onnxruntime rembg gradio "numpy==1.26.4" "pillow==10.3.0" "opencv-python-headless==4.8.1.78" "opencv-python==4.8.1.78" --quiet

print("✅ INSTALLATION COMPLETE! Now you MUST go to the top menu -> Runtime -> Restart session. After restarting, run Cell 2.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 104.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 83.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.1/49.1 MB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 MB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 59.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 47.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 113.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.9/54.9 kB

In [ ]:
# CELL 2: THE APP CODE
# (Run this ONLY after restarting the runtime!)
import gradio as gr
from diffusers import StableDiffusionPipeline
import torch
from PIL import Image, ImageEnhance, ImageFilter, ImageOps, ImageChops
from rembg import remove
import cv2
import shutil
import numpy as np
import warnings, os
warnings.filterwarnings("ignore")

# --------------------------
# Gradient Dashboard Style
# --------------------------
GRADIENT_CSS = """
.gradio-container {
    background: #f0f4f9 !important;
    font-family: 'Segoe UI', sans-serif !important;
    color: white !important;
}

/* Top Title */
.top-title {
    background: linear-gradient(90deg, #42a5f5, #1e88e5) !important;
    color: white !important;
    font-size: 2.2rem !important;
    font-weight: bold !important;
    text-align: center !important;
    border-radius: 14px !important;
    padding: 18px !important;
    margin-bottom: 20px !important;
    box-shadow: 0 4px 15px rgba(0,0,0,0.25) !important;
}

/* Left Panel */
.left-panel {
    background: linear-gradient(180deg, #ffcc80, #ffb74d) !important;
    border-radius: 14px !important;
    padding: 12px !important;
    color: white !important;
}

/* Right Panel */
.right-panel {
    background: linear-gradient(180deg, #90caf9, #64b5f6) !important;
    border-radius: 14px !important;
    padding: 12px !important;
    color: white !important;
}

/* Small Card */
.section-card {
    background: rgba(255, 255, 255, 0.2) !important;
    border-radius: 8px !important;
    padding: 8px !important;
    margin-bottom: 10px !important;
    box-shadow: 0 2px 5px rgba(0,0,0,0.08) !important;
    color: #000 !important;
}

/* Section Title */
.section-title {
    font-size: 1.05rem !important;
    font-weight: bold !important;
    color: white !important;
    margin-bottom: 6px !important;
    text-shadow: 1px 1px 3px rgba(0,0,0,0.3) !important;
}

/* Generate Button */
.gen-button {
    background: linear-gradient(90deg, #f57c00, #ef6c00) !important;
    color: white !important;
    font-weight: bold !important;
    border-radius: 8px !important;
    padding: 10px !important;
    width: 100% !important;
    border: none !important;
}

/* Download Button (Matching the Blue Theme) */
.download-btn {
    background: linear-gradient(90deg, #42a5f5, #1e88e5) !important;
    color: white !important;
    font-weight: bold !important;
    border-radius: 8px !important;
    padding: 10px !important;
    width: 100% !important;
    text-align: center !important;
    border: none !important;
}
"""

# --------------------------
# Utility Functions
# --------------------------
def clean_alpha(img, threshold=10):
    img = img.convert("RGBA")
    arr = np.array(img)
    arr[arr[:,:,3] < threshold] = [0,0,0,0]
    return Image.fromarray(arr, "RGBA")

def feather_edges(img, radius=3):
    r,g,b,a = img.split()
    a_blur = a.filter(ImageFilter.GaussianBlur(radius))
    a = ImageChops.blend(a, a_blur, 0.7)
    return Image.merge("RGBA", (r,g,b,a))

def enhance_image(img):
    img = img.convert("RGB")
    img = ImageOps.autocontrast(img, cutoff=1)
    img = ImageEnhance.Color(img).enhance(1.1)
    img = ImageEnhance.Sharpness(img).enhance(1.05)
    return img.convert("RGBA")

def color_match(source, target):
    try:
        src = cv2.cvtColor(np.array(source.convert("RGB")), cv2.COLOR_RGB2LAB).astype("float32")
        tgt = cv2.cvtColor(np.array(target.convert("RGB")), cv2.COLOR_RGB2LAB).astype("float32")
        (lmS,amS,bmS),(lsS,asS,bsS) = cv2.meanStdDev(src)
        (lmT,amT,bmT),(lsT,asT,bsT) = cv2.meanStdDev(tgt)
        l,a,b = cv2.split(tgt)
        l=(l-lmT[0])*(lsS[0]/(lsT[0]+1e-6))+lmS[0]
        a=(a-amT[0])*(asS[0]/(asT[0]+1e-6))+amS[0]
        b=(b-bmT[0])*(bsS[0]/(bsT[0]+1e-6))+bmS[0]
        out = cv2.merge([l,a,b])
        out = np.clip(out,0,255)
        out = cv2.cvtColor(out.astype("uint8"),cv2.COLOR_LAB2RGB)
        return Image.fromarray(out).convert("RGB")
    except:
        return target.convert("RGB")

def composite_images(fg, bg):
    bg = bg.resize(fg.size, Image.Resampling.LANCZOS).convert("RGB")
    fg = fg.convert("RGBA")
    arr_fg = np.array(fg)
    arr_bg = np.array(bg)
    alpha = arr_fg[:,:,3]/255.0
    alpha = cv2.GaussianBlur(alpha,(3,3),1.0)
    alpha_3 = np.dstack([alpha]*3)
    out = arr_fg[:,:,:3]*alpha_3 + arr_bg*(1-alpha_3)
    return Image.fromarray(out.astype(np.uint8),"RGB")

# --------------------------
# Load Model (Stable Diffusion 1.5)
# --------------------------

device = "cuda" if torch.cuda.is_available() else "cpu"
pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16 if device=="cuda" else torch.float32
).to(device)
pipe.enable_attention_slicing()

# --------------------------
# Main Function
# --------------------------
def generate_with_bg(uploaded_img, prompt, bg_upload_img, resolution=512, steps=25):
    if uploaded_img is None:
        return None, None

    # Step 1: Choose background
    if bg_upload_img is not None:
        bg = bg_upload_img.resize((resolution, resolution))
    elif prompt.strip():
        bg = pipe(
            prompt=f"{prompt}, ultra realistic, cinematic lighting",
            negative_prompt="blurry, distorted, low quality",
            num_inference_steps=steps,
            guidance_scale=7.0,
            width=resolution,
            height=resolution
        ).images[0]
    else:
        return None, None

    # Step 2: Process foreground
    fg = enhance_image(uploaded_img)
    fg = remove(fg).convert("RGBA")
    fg = clean_alpha(fg)
    fg = feather_edges(fg, 2)

    # Step 3: Composite
    result = composite_images(fg, bg)

    # Step 4: Save to temp path and copy for download
    temp_path = "/tmp/result_temp.png"
    result.save(temp_path)
    download_path = "result.png"
    shutil.copy(temp_path, download_path)

    return result, download_path

# --------------------------
# Gradio Interface
# --------------------------
with gr.Blocks(title="AI Background Studio", css=GRADIENT_CSS) as demo:
    gr.HTML('<div class="top-title">🎨 AI Background Studio</div>')

    with gr.Row(equal_height=True):
        # Left Panel
        with gr.Column(scale=1, elem_classes="left-panel"):
            with gr.Group(elem_classes="section-card"):
                gr.HTML('<div class="section-title">📤 Upload Image</div>')
                img_input = gr.Image(type="pil", label="Upload Foreground Image", height=280)

            with gr.Group(elem_classes="section-card"):
                gr.HTML('<div class="section-title">📝 Background Options</div>')
                prompt_input = gr.Textbox(placeholder="e.g., futuristic office with neon lights",
                                           lines=2, label="Background Prompt")
                bg_upload = gr.Image(type="pil", label="Or Upload Background", height=180)

            with gr.Group(elem_classes="section-card"):
                generate_btn = gr.Button("✨ Generate", elem_classes="gen-button")

        # Right Panel
        with gr.Column(scale=1, elem_classes="right-panel"):
            with gr.Group(elem_classes="section-card"):
                gr.HTML('<div class="section-title">🖼️ Final Result</div>')
                img_output = gr.Image(type="pil", label="Result Image", height=280, show_download_button=True)

            with gr.Group(elem_classes="section-card"):
                download_btn = gr.File(label="⬇️ Download Final Result", elem_classes="download-btn")

    # ✅ Only keep the actual needed inputs/outputs
    generate_btn.click(
        fn=generate_with_bg,
        inputs=[img_input, prompt_input, bg_upload],
        outputs=[img_output, download_btn]
    )

if __name__ == "__main__":
    demo.launch(debug=True)

model_index.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

merges.txt: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/617 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

vae/diffusion_pytorch_model.safetensors:   0%|          | 0.00/335M [00:00<?, ?B/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://88a30391e023caca08.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.


KeyboardInterrupt: 